# 02 模型训练
**前置条件**：先运行 `01_data_processing.ipynb`

---
## 📌 模型体系概览

| 模型 | 类型 | 核心假设 | 适用场景 |
|------|------|---------|----------|
| **PLS** | 线性潜变量 | 过程变量与质量指标存在线性关联 | 工业基线，多重共线性 |
| **XGBoost** | 梯度提升树 | 非线性关系，特征交叉作用 | 最优精度，标准场景 |
| **LightGBM** | 梯度提升树 | 同XGBoost，更快 | 数据量大时优选 |
| **LSTM** | 深度时序 | 质量指标有动态时延依赖 | 时延明显时 |
| **Stacking** | 元学习集成 | 不同模型偏差互补 | 生产部署优选 |

**训练策略**：时序交叉验证（`TimeSeriesSplit`）——不允许未来数据出现在验证集，模拟生产在线预测场景。

## 1. 加载数据与环境

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml, joblib

sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams.update({'figure.figsize': (12, 4), 'figure.dpi': 100})
sns.set_theme(style='whitegrid')
Path('../outputs/models').mkdir(parents=True, exist_ok=True)
Path('../outputs/figures').mkdir(parents=True, exist_ok=True)

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import TimeSeriesSplit

with open('../config.yaml', 'r', encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
print('配置加载完成')

In [ ]:
X_train = np.load('../outputs/processed/X_train.npy')
X_test  = np.load('../outputs/processed/X_test.npy')
y_train = np.load('../outputs/processed/y_train.npy')
y_test  = np.load('../outputs/processed/y_test.npy')
selected_names = joblib.load('../outputs/processed/selected_names.pkl')
print(f'训练集: X={X_train.shape}  y={y_train.shape}')
print(f'测试集: X={X_test.shape}   y={y_test.shape}')
print(f'特征数: {len(selected_names)}  |  y_train范围: [{y_train.min():.1f}, {y_train.max():.1f}] °C')

In [ ]:
def cv_metrics(model, X, y, n_splits=5):
    """时序交叉验证，每折只能用历史数据训练，不能用未来数据。"""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    results = []
    for fold, (tr, va) in enumerate(tscv.split(X)):
        model.fit(X[tr], y[tr])
        pred = model.predict(X[va])
        if len(pred) != len(va): pred = pred[-len(va):]
        results.append({'fold': fold+1,
                        'RMSE': np.sqrt(mean_squared_error(y[va], pred)),
                        'MAE':  mean_absolute_error(y[va], pred),
                        'R2':   r2_score(y[va], pred)})
    df = pd.DataFrame(results).set_index('fold')
    df.loc['mean'] = df.mean(); df.loc['std'] = df.std()
    return df.round(4)

all_cv_results = {}  # 汇总各模型CV结果
trained_models = {}  # 保存最终模型
val_size = max(int(len(X_train) * 0.15), 10)
X_tr, X_va = X_train[:-val_size], X_train[-val_size:]
y_tr, y_va = y_train[:-val_size], y_train[-val_size:]
print(f'工具函数定义完成  |  early stopping验证集大小: {val_size}')

> ### 📐 原理：为什么必须用时序交叉验证？
> 
> 普通K-Fold交叉验证随机打乱数据，会导致**未来数据泄露到训练集**（验证集早于训练集的数据点）。
> 在时序场景中这会造成验证指标虚高，模型在生产中表现远差于验证集。
> 
> `TimeSeriesSplit`保证每折验证集**时间上晚于**训练集，精确模拟在线预测场景：
> ```
> Fold 1: Train=[1..200]  Val=[201..240]
> Fold 2: Train=[1..240]  Val=[241..280]
> Fold 3: Train=[1..280]  Val=[281..320]
> ...```

## 2. PLS（偏最小二乘）

> ### 📐 原理详解
> 
> **问题**：DCS温度信号高度共线（相邻塔板温差~1°C，相关系数>0.99），直接做线性回归系数不稳定。
> 
> **PLS解法**：
> 1. 寻找潜变量矩阵 $T = XW^*$，$U = YC$
> 2. 目标：最大化 $\text{cov}^2(t_h, u_h) = \text{cov}^2(Xw_h, Yc_h)$
> 3. 每个成分提取后，从X和Y中去除该成分的信息（deflation），再提取下一个
> 
> **优势**：
> - 自动压缩共线性，等价于在潜变量空间回归
> - 只需少量成分（`n_components`~5-15）即可捕捉主要变化
> - 化工软测量40年工业标准，可解释性强
> 
> **超参数选择**：`n_components` 通过时序CV搜索，选CV R²最高的值

In [ ]:
from sklearn.cross_decomposition import PLSRegression

n_comp_range = [n for n in cfg['models']['pls'].get('n_components_range', [5,8,10,12,15])
                if n < min(X_train.shape)]
tscv = TimeSeriesSplit(n_splits=5)
comp_scores = []
for n in n_comp_range:
    fold_r2 = []
    for tr, va in tscv.split(X_train):
        m = PLSRegression(n_components=n)
        m.fit(X_train[tr], y_train[tr])
        p = m.predict(X_train[va]).ravel()
        fold_r2.append(r2_score(y_train[va], p))
    comp_scores.append({'n_components': n, 'CV_R2_mean': np.mean(fold_r2), 'CV_R2_std': np.std(fold_r2)})

comp_df = pd.DataFrame(comp_scores).set_index('n_components')
print(comp_df.round(4))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(comp_df.index, comp_df['CV_R2_mean'], yerr=comp_df['CV_R2_std'],
            marker='o', capsize=4, color='steelblue')
ax.set_xlabel('n_components（潜变量数）'); ax.set_ylabel('CV R²')
ax.set_title('PLS n_components 选择（误差棒=跨折标准差）')
plt.tight_layout(); plt.show()

best_n = comp_df['CV_R2_mean'].idxmax()
print(f'\n最优 n_components = {best_n}  (CV R²={comp_df.loc[best_n, "CV_R2_mean"]:.4f})')

In [ ]:
pls = PLSRegression(n_components=best_n)
cv_pls = cv_metrics(pls, X_train, y_train)
all_cv_results['PLS'] = cv_pls
print('=== PLS 时序交叉验证 ===')
print(cv_pls)

pls.fit(X_train, y_train)
trained_models['pls'] = pls

# PLS VIP可视化
W = pls.x_weights_; T = pls.x_scores_; Q = pls.y_loadings_
ss = np.sum(T**2, axis=0) * np.sum(Q**2, axis=0)
W_norm = W / np.linalg.norm(W, axis=0, keepdims=True)
vip = np.sqrt(W.shape[0] * np.sum(W_norm**2 * ss, axis=1) / (ss.sum() + 1e-9))
vip_df = pd.Series(vip, index=selected_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
top_vip = vip_df.head(20)
top_vip.plot(kind='bar', ax=ax,
             color=['tomato' if v >= 1.0 else 'steelblue' for v in top_vip])
ax.axhline(1.0, color='red', linestyle='--', label='VIP=1.0（重要性阈值）')
ax.set_title('PLS VIP Score Top-20')
ax.set_xticklabels([n.replace('CDU2.','') for n in top_vip.index], rotation=45, ha='right')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/pls_vip.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'VIP≥1.0的变量数: {(vip>=1.0).sum()}')

## 3. XGBoost

> ### 📐 原理详解
> 
> **梯度提升框架**：集成 $M$ 棵弱学习器（回归树），逐步减小残差：
> $$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$
> 
> **XGBoost创新**：目标函数加入二阶泰勒展开 + 树复杂度正则：
> $$\mathcal{L}^{(m)} = \sum_i [g_i f_m(x_i) + \frac{1}{2} h_i f_m^2(x_i)] + \Omega(f_m)$$
> 其中 $g_i = \partial_{\hat{y}^{(m-1)}} l$（一阶梯度），$h_i = \partial^2_{\hat{y}^{(m-1)}} l$（二阶梯度）
> 
> **关键超参数**：
> - `max_depth`: 树深度，控制单棵树的复杂度（越大越易过拟合）
> - `learning_rate` (η): 步长，越小需要更多树，但泛化更好
> - `subsample` / `colsample_bytree`: 行列采样，增加多样性防过拟合
> - `early_stopping_rounds`: 验证集误差不再改善时自动停止训练

In [ ]:
from xgboost import XGBRegressor
xgb_cfg = cfg['models']['xgboost']

xgb = XGBRegressor(
    n_estimators=xgb_cfg.get('n_estimators', 500),
    max_depth=xgb_cfg.get('max_depth', 5),
    learning_rate=xgb_cfg.get('learning_rate', 0.03),
    subsample=xgb_cfg.get('subsample', 0.8),
    colsample_bytree=xgb_cfg.get('colsample_bytree', 0.8),
    min_child_weight=xgb_cfg.get('min_child_weight', 3),
    reg_alpha=xgb_cfg.get('reg_alpha', 0.1),
    reg_lambda=xgb_cfg.get('reg_lambda', 1.0),
    early_stopping_rounds=xgb_cfg.get('early_stopping_rounds', 50),
    random_state=42, verbosity=0,
)
xgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
print(f'Early stopping最优树数: {xgb.best_iteration}')

xgb_cv_model = XGBRegressor(n_estimators=xgb.best_iteration,
                             max_depth=xgb_cfg.get('max_depth', 5),
                             learning_rate=xgb_cfg.get('learning_rate', 0.03),
                             subsample=xgb_cfg.get('subsample', 0.8),
                             colsample_bytree=xgb_cfg.get('colsample_bytree', 0.8),
                             random_state=42, verbosity=0)
cv_xgb = cv_metrics(xgb_cv_model, X_train, y_train)
all_cv_results['XGBoost'] = cv_xgb
print('\n=== XGBoost 时序交叉验证 ===')
print(cv_xgb)

xgb.fit(X_train, y_train)
trained_models['xgboost'] = xgb

In [ ]:
imp_xgb = pd.Series(xgb.feature_importances_, index=selected_names).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 5))
imp_xgb.head(20).plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('XGBoost 特征重要度 Top-20（基于分裂增益）')
ax.set_xticklabels([n.replace('CDU2.','') for n in imp_xgb.head(20).index], rotation=45, ha='right')
ax.set_ylabel('Importance')
plt.tight_layout()
plt.savefig('../outputs/figures/xgboost_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. LightGBM

> ### 📐 原理：与XGBoost的关键差异
> 
> | 特性 | XGBoost | LightGBM |
> |------|---------|----------|
> | 生长策略 | **节点优先**（level-wise，同层所有节点扩展）| **叶节点优先**（leaf-wise，选增益最大叶） |
> | 特征处理 | 精确分割点 | **直方图算法**（连续值分桶，大幅加速）|
> | 速度 | 基准 | 快3-5倍 |
> | 过拟合风险 | 低 | 叶节点优先易深度生长，需控制`num_leaves` |
> 
> **叶节点优先生长**：每次分裂使损失减少最多的叶节点，
> 能以更少树达到相同精度，但需要 `num_leaves` 和 `min_child_samples` 控制复杂度。

In [ ]:
import lightgbm as lgb
lgb_cfg = cfg['models']['lightgbm']

callbacks = [lgb.early_stopping(lgb_cfg.get('early_stopping_rounds', 50), verbose=False),
             lgb.log_evaluation(period=-1)]

lgbm = lgb.LGBMRegressor(
    n_estimators=lgb_cfg.get('n_estimators', 500),
    max_depth=lgb_cfg.get('max_depth', 5),
    learning_rate=lgb_cfg.get('learning_rate', 0.03),
    num_leaves=lgb_cfg.get('num_leaves', 31),
    subsample=lgb_cfg.get('subsample', 0.8),
    colsample_bytree=lgb_cfg.get('colsample_bytree', 0.8),
    min_child_samples=lgb_cfg.get('min_child_samples', 10),
    reg_alpha=lgb_cfg.get('reg_alpha', 0.1),
    reg_lambda=lgb_cfg.get('reg_lambda', 1.0),
    random_state=42, verbosity=-1,
)
lgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=callbacks)
print(f'Early stopping最优迭代: {lgbm.best_iteration_}')

lgbm_cv_model = lgb.LGBMRegressor(n_estimators=lgbm.best_iteration_,
                                   max_depth=lgb_cfg.get('max_depth', 5),
                                   learning_rate=lgb_cfg.get('learning_rate', 0.03),
                                   num_leaves=lgb_cfg.get('num_leaves', 31),
                                   random_state=42, verbosity=-1)
cv_lgbm = cv_metrics(lgbm_cv_model, X_train, y_train)
all_cv_results['LightGBM'] = cv_lgbm
print('\n=== LightGBM 时序交叉验证 ===')
print(cv_lgbm)

lgbm.fit(X_train, y_train)
trained_models['lightgbm'] = lgbm

# 双模型重要度对比
imp_lgbm = pd.Series(lgbm.feature_importances_, index=selected_names).sort_values(ascending=False)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (name, imp) in zip(axes, [('XGBoost', imp_xgb), ('LightGBM', imp_lgbm)]):
    imp.head(15).plot(kind='bar', ax=ax, color='steelblue')
    ax.set_title(f'{name} 特征重要度 Top-15')
    ax.set_xticklabels([n.replace('CDU2.','') for n in imp.head(15).index], rotation=45, ha='right')
plt.tight_layout()
plt.savefig('../outputs/figures/tree_importance_compare.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. LSTM（可选，耗时较长）

> ### 📐 原理：为什么蒸馏过程适合用LSTM？
> 
> 蒸馏塔存在明显**传输时延**：加热炉出口温度变化 → 经过塔盘传热传质 → 塔顶产品质量变化，
> 这个过程有5-20分钟延迟。普通静态模型（PLS/XGBoost）用单时间点输入，无法建模这个动态过程。
> 
> **LSTM门控机制**（以单时间步为例）：
> - 遗忘门 $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$：决定丢弃多少历史
> - 输入门 $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$：决定写入多少新信息
> - 单元状态 $C_t = f_t \odot C_{t-1} + i_t \odot \tanh(W_C [h_{t-1}, x_t] + b_C)$
> - 输出门 $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$，$h_t = o_t \odot \tanh(C_t)$
> 
> **输入窗口**：取最近`sequence_length`个时间步的过程数据作为序列输入，
> LSTM自动学习哪些历史时刻对当前预测最重要。
> 
> 如不需LSTM可将 `TRAIN_LSTM = False` 跳过此节。

In [ ]:
TRAIN_LSTM = cfg['models']['lstm'].get('enabled', True)
lstm_cfg   = cfg['models']['lstm']
SEQ_LEN    = lstm_cfg.get('sequence_length', 12)
print(f'LSTM训练: {"开启" if TRAIN_LSTM else "跳过"}  |  序列窗口长度: {SEQ_LEN}')

In [ ]:
if TRAIN_LSTM:
    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    def make_sequences(X, y=None, seq_len=SEQ_LEN):
        seqs, targets = [], []
        for i in range(seq_len, len(X)):
            seqs.append(X[i-seq_len:i])
            if y is not None: targets.append(y[i])
        seqs = np.array(seqs, dtype=np.float32)
        return (seqs, np.array(targets, dtype=np.float32)) if y is not None else seqs

    seqs, tgts = make_sequences(X_train, y_train)
    va_size = max(int(len(seqs) * 0.15), 5)
    X_seq_tr, X_seq_va = seqs[:-va_size], seqs[-va_size:]
    y_seq_tr, y_seq_va = tgts[:-va_size], tgts[-va_size:]

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'设备: {device}')

    class SoftSensorLSTM(nn.Module):
        def __init__(self, n_feat, hidden, n_layers, dropout):
            super().__init__()
            self.lstm = nn.LSTM(n_feat, hidden, n_layers, batch_first=True,
                                dropout=dropout if n_layers > 1 else 0)
            self.fc = nn.Linear(hidden, 1)
        def forward(self, x):
            out, _ = self.lstm(x)
            return self.fc(out[:, -1, :]).squeeze(-1)

    net = SoftSensorLSTM(X_train.shape[1],
                         lstm_cfg.get('hidden_size', 64),
                         lstm_cfg.get('num_layers', 2),
                         lstm_cfg.get('dropout', 0.2)).to(device)

    loader  = DataLoader(TensorDataset(torch.tensor(X_seq_tr).to(device),
                                       torch.tensor(y_seq_tr).to(device)),
                         batch_size=lstm_cfg.get('batch_size', 32), shuffle=False)
    X_va_t  = torch.tensor(X_seq_va).to(device)
    y_va_t  = torch.tensor(y_seq_va).to(device)
    opt     = torch.optim.Adam(net.parameters(), lr=lstm_cfg.get('learning_rate', 0.001))
    sched   = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=5, factor=0.5)
    loss_fn = nn.MSELoss()
    PATIENCE = lstm_cfg.get('patience', 15)
    EPOCHS   = lstm_cfg.get('epochs', 100)

    train_losses, val_losses = [], []
    best_val, no_imp, best_state = np.inf, 0, None
    for ep in range(EPOCHS):
        net.train(); ep_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            l = loss_fn(net(xb), yb); l.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step(); ep_loss += l.item()
        train_losses.append(ep_loss / len(loader))
        net.eval()
        with torch.no_grad(): vl = loss_fn(net(X_va_t), y_va_t).item()
        val_losses.append(vl); sched.step(vl)
        if vl < best_val:
            best_val, no_imp = vl, 0
            best_state = {k: v.cpu().clone() for k, v in net.state_dict().items()}
        else:
            no_imp += 1
            if no_imp >= PATIENCE:
                print(f'Early stop @ epoch {ep+1}'); break
        if (ep+1) % 10 == 0:
            print(f'Epoch {ep+1:3d}  train={train_losses[-1]:.4f}  val={vl:.4f}')

    net.load_state_dict(best_state)
    trained_models['lstm'] = {'net': net, 'seq_len': SEQ_LEN, 'device': device}

    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(train_losses, label='Train Loss'); ax.plot(val_losses, label='Val Loss')
    ax.set_title('LSTM 训练曲线（MSE Loss）'); ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/figures/lstm_loss.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'LSTM训练完成  best_val_RMSE={np.sqrt(best_val):.3f}°C')
else:
    print('跳过LSTM')

## 6. Stacking 集成

> ### 📐 原理：为什么Stacking比简单平均更优？
> 
> **问题**：直接平均各模型预测，对所有模型等权，无法利用各模型在不同样本上的优劣差异。
> 
> **Stacking两级架构**：
> - **Level 0（基模型）**：PLS、XGBoost、LightGBM 各自预测
> - **Level 1（元学习器）**：用各模型的OOF（Out-Of-Fold）预测作为特征，训练Ridge回归
> 
> **关键：OOF防止过拟合**
> 元学习器的训练数据来自基模型在**验证集**上的预测（不是训练集），
> 避免基模型过拟合导致元学习器学到无效信号。
> 
> **Ridge元学习器**：$\hat{y} = \alpha_1 \hat{y}_{PLS} + \alpha_2 \hat{y}_{XGB} + \alpha_3 \hat{y}_{LGB}$，
> 系数由Ridge（L2正则）回归自动学习，$\alpha_i$ 可为负（纠正某模型的系统偏差）。

In [ ]:
from sklearn.linear_model import Ridge

base_models = {'PLS': trained_models['pls'],
               'XGBoost': trained_models['xgboost'],
               'LightGBM': trained_models['lightgbm']}

tscv = TimeSeriesSplit(n_splits=5)
oof_preds = np.zeros((len(X_train), len(base_models)))

for fold, (tr, va) in enumerate(tscv.split(X_train)):
    for j, (name, model) in enumerate(base_models.items()):
        model.fit(X_train[tr], y_train[tr])
        p = model.predict(X_train[va])
        oof_preds[va, j] = p if len(p) == len(va) else p[-len(va):]

meta = Ridge(alpha=1.0)
meta.fit(oof_preds, y_train)

print('元学习器权重（Ridge回归系数）:')
for name, coef in zip(base_models.keys(), meta.coef_):
    print(f'  {name}: {coef:.4f}')
print(f'截距: {meta.intercept_:.4f}')

for m in base_models.values():
    m.fit(X_train, y_train)

trained_models['stacking'] = {'base': {k.lower(): v for k, v in base_models.items()}, 'meta': meta}
print('\nStacking集成构建完成')

## 7. 交叉验证结果对比

In [ ]:
summary = []
for model_name, cv_df in all_cv_results.items():
    row = cv_df.loc['mean'].to_dict(); row['Model'] = model_name
    summary.append(row)
summary_df = pd.DataFrame(summary).set_index('Model')
print('=' * 50)
print('       交叉验证均值对比')
print('=' * 50)
print(summary_df.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7']
for ax, metric in zip(axes, ['RMSE', 'MAE', 'R2']):
    bars = ax.bar(summary_df.index, summary_df[metric], color=colors[:len(summary_df)], edgecolor='white')
    ax.set_title(f'CV {metric}'); ax.tick_params(axis='x', rotation=30)
    for bar, val in zip(bars, summary_df[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig('../outputs/figures/cv_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. 保存模型

In [ ]:
save_dir = Path('../outputs/models')
joblib.dump(trained_models['pls'],      save_dir / 'pls.pkl')
joblib.dump(trained_models['xgboost'],  save_dir / 'xgboost.pkl')
joblib.dump(trained_models['lightgbm'], save_dir / 'lightgbm.pkl')
joblib.dump(trained_models['stacking'], save_dir / 'stacking.pkl')

if 'lstm' in trained_models:
    import torch
    torch.save(trained_models['lstm']['net'].state_dict(), save_dir / 'lstm_state.pt')
    joblib.dump({'seq_len': trained_models['lstm']['seq_len'],
                 'n_feat': X_train.shape[1],
                 'hidden': lstm_cfg.get('hidden_size', 64),
                 'n_layers': lstm_cfg.get('num_layers', 2)},
                save_dir / 'lstm_config.pkl')

print('模型保存完成:')
for f in sorted(save_dir.iterdir()):
    print(f'  {f.name}  ({f.stat().st_size/1024:.1f} KB)')
print('\n➡  请运行 03_prediction_evaluation.ipynb')